# G-Portal catalogue search — retired endpoint

> ⚠️ **Status: this example cannot run live.** JAXA decommissioned the G-Portal
> **CSW catalogue-search** endpoint. The `gportal` SDK's `BASE_URL` now
> permanently (`301`) redirects every `/csw/csw` request to the portal landing
> page, so a search returns HTML instead of JSON and the SDK fails to parse it.
> Because the `gportal` protocol relies on this search to enumerate products,
> **both** the anonymous search *and* the credentialed SFTP download are blocked
> until a successor endpoint is found. Tracked in earthlens issue **#1415**.

This notebook used to walk the full credentialed path: pick a GCOM-C/SGLI L3
ocean product from the catalog, search the live G-Portal catalogue for a 1-day
window, download the matching products over SFTP, and inspect the HDF5. The
catalog half still works and is kept below; the live search/download is
documented rather than executed. For a JAXA example that *does* run end-to-end
today, see [`02_aw3d30_quickstart.ipynb`](02_aw3d30_quickstart.ipynb) — the
authless `jaxa-earth` (STAC) path is unaffected by the CSW retirement.

## Open the catalog and pick a product

In [ ]:
from earthlens.jaxa import Catalog

cat = Catalog()
row = cat.get('sgli-l3-nwlr')
print(row)
print('aliases:      ', row.aliases)
print('protocol:     ', row.protocol)
print('short_name:   ', row.short_name)

## What the search used to do — and what happens now

`gportal.search()` is anonymous (credentials are only needed for the SFTP
download). It *used* to query the live G-Portal CSW catalogue:

```python
import gportal

result = gportal.search(
    dataset_ids=[row.short_name],
    start_time="2024-01-01",
    end_time="2024-01-02",
    count=3,
)
print(result.matched())
```

That request now `301`-redirects to `https://www.gportal.jaxa.jp/`; the SDK
follows the redirect, lands on the ~1 MB HTML landing page, and
`response.json()` fails with `JSONDecodeError`. Driven through the earthlens
backend —

```python
from earthlens.core import EarthLens

EarthLens(data_source="jaxa", variables=["sgli-l3-nwlr"], ...).download()
```

— the branch confirms the retirement with a no-redirect probe of the service
URL and raises `GportalEndpointRetiredError`: a `GportalUnavailableError`
subclass (so existing `except GportalUnavailableError` handlers still catch it)
that is distinct, so you can tell *"the endpoint is gone"* from *"the service is
temporarily down"*. The cell below shows that relationship without touching the
network.

In [ ]:
from earthlens.jaxa import (
    GportalEndpointRetiredError,
    GportalUnavailableError,
)

# The retirement error subclasses the transient-outage error, so old handlers
# keep working, but you can branch on the narrower type to tell a permanently
# gone endpoint from a service that is merely down right now.
is_subclass = issubclass(GportalEndpointRetiredError, GportalUnavailableError)
print("retired is-a unavailable:", is_subclass)
print("retired error type:      ", GportalEndpointRetiredError.__name__)

## A JAXA example that runs today

The authless `jaxa-earth` (STAC) protocol is unaffected by the CSW retirement.
See [`02_aw3d30_quickstart.ipynb`](02_aw3d30_quickstart.ipynb) for a JAXA
download that works end-to-end today, and the
[JAXA usage reference](../../reference/jaxa/usage.md) for the full request shape
and the current status of the `gportal` protocol.